# Week 6: Language Models
AI for Good - Minor ADSAI

[Open this notebook in Google Colab](https://colab.research.google.com/github/BTuyn/AI4G/blob/main/Week%206/Week6_Workshop_Student.ipynb) - nothing to install, everything
runs in your browser. If you opened it in Colab already: great.

Today you meet the three ideas behind every modern language model, one at a
time, each one fixing a problem of the one before:

| Step | Model | What it adds |
|------|-------|--------------|
| 1 | **Word embeddings** | words become numbers that carry meaning |
| 2 | **RNN**, then **LSTM** | reading in order, and remembering what you read |
| 3 | **Transformer** | looking at every word at once, so context decides the meaning |

You do **not** build or train any of these today. You use ready-made models
and look at what they do well, where they break, and who gets hurt when they
break. The spine of the week is the same as in Weeks 4 and 5:
**text -> numbers -> model -> prediction**.

Today runs in **three learning waves**, each starting with a short
mini-lecture. A wave starts at every horizontal line in this notebook,
marked like this:

**LEARNING WAVE 1 · SLIDES 4-8**

- Wave 1 - section 1: words as numbers (embeddings)
- Wave 2 - section 2: order and memory (RNN and LSTM)
- Wave 3 - section 3: context and attention (transformers)
- Then section 4: the big individual assignment, and section 5: wrap-up

Do not run ahead into the next wave - wait for the mini-lecture.

### 0. Setup - run these two cells once

Nothing to write here. Run each cell with Shift + Enter and wait until it
finishes (the first one downloads a 66 MB file, a few seconds). If a cell
prints a long warning about a missing token or about "unauthenticated
requests", ignore it.

In [ ]:
# Run, do not read: downloads the pre-trained word embeddings (GloVe, trained by Stanford)
import urllib.request, gzip
import numpy as np
import matplotlib.pyplot as plt

url = "https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz"
urllib.request.urlretrieve(url, "glove50.gz")

words = []
vectors = []
with gzip.open("glove50.gz", "rt", encoding="utf-8") as f:
    f.readline()                       # first line is just file info
    for i, line in enumerate(f):
        if i >= 50000:                 # keep the 50,000 most common words
            break
        parts = line.rstrip().split(" ")
        words.append(parts[0])
        vectors.append(np.array(parts[1:], dtype="float32"))

V = np.vstack(vectors)
V = V / np.linalg.norm(V, axis=1, keepdims=True)   # length 1, so a dot product = similarity
index = {}
for i, w in enumerate(words):
    index[w] = i
print("Loaded", len(words), "words, each one a list of", V.shape[1], "numbers")

In [ ]:
# Run, do not read: small helper functions that we use all day
def vector(word):
    return V[index[word]]

def similarity(a, b):
    # 1.0 = used in exactly the same way, 0 = nothing in common
    return round(float(vector(a) @ vector(b)), 2)

def neighbours(word, n=8):
    scores = V @ vector(word)
    best = np.argsort(-scores)[1:n + 1]
    return [(words[i], round(float(scores[i]), 2)) for i in best]

def analogy(a, b, c, n=5):
    # "a is to b as c is to ?"   (for example: man is to king as woman is to ?)
    target = vector(b) - vector(a) + vector(c)
    scores = V @ target
    result = []
    for i in np.argsort(-scores):
        if words[i] not in (a, b, c):
            result.append(words[i])
        if len(result) == n:
            break
    return result

def plot_words(word_list):
    # squash the 50 numbers per word into 2, so we can draw a map
    X = np.array([vector(w) for w in word_list])
    X = X - X.mean(axis=0)
    u, s, vt = np.linalg.svd(X, full_matrices=False)
    xy = X @ vt[:2].T
    plt.figure(figsize=(8, 6))
    plt.scatter(xy[:, 0], xy[:, 1])
    for k, w in enumerate(word_list):
        plt.text(xy[k, 0] + 0.02, xy[k, 1] + 0.02, w, fontsize=11)
    plt.title("Words on a map (50 numbers squashed into 2)")
    plt.axis("off")
    plt.show()

---

**LEARNING WAVE 1 · SLIDES 4-8**

## 1. Words as numbers

Sections 1.1 to 1.7. From counting words to giving every word a position on a map of meaning.

### 1.1 What is an embedding?

In slide 5 a sentence became a row of word counts. A count tells you *which*
words appear, but not what they *mean*: to a count, `dog` and `cat` are as
unrelated as `dog` and `car`.

An **embedding** fixes that. Every word gets a list of numbers, and words
that are used in similar ways get similar numbers. The file you just loaded
was trained by Stanford on billions of words. It is **pre-trained**: you do
not train anything today.

Run the cell below. It shows the list of numbers for the word `clinic`.

In [ ]:
print(vector("clinic")[:10])
print("This word is a list of", len(vector("clinic")), "numbers. We only printed the first 10.")

Your answer:

The word `clinic` is just a list of 50 numbers. One number alone means nothing. The meaning is in the whole list, and words that are used in the same way get lists that look alike.


### 1.2 How close are two words?

`similarity(a, b)` compares the two lists of numbers. It gives **1.0** when
two words are used in exactly the same way and values near **0** when they
have nothing in common.

**Predict first.** Which of these five pairs will be the most similar, and
which the least?

- `dog` and `cat`
- `dog` and `car`
- `doctor` and `nurse`
- `clinic` and `hospital`
- `king` and `banana`

Your prediction:

Most similar: `dog` and `cat` (both pets). Least similar: `king` and `banana` (nothing in common).


In [ ]:
pairs = [("dog", "cat"), ("dog", "car"), ("doctor", "nurse"), ("clinic", "hospital"), ("king", "banana")]
for pair in pairs:
    print(pair[0], "-", pair[1], ":", similarity(pair[0], pair[1]))

Your answer:

My prediction was right. `dog`-`cat` is the highest (about 0.92) and `king`-`banana` the lowest (about 0.22). `clinic`-`hospital` (0.86) and `doctor`-`nurse` (0.80) are also high, because they appear in the same kind of texts. `dog`-`car` is in the middle (0.46): the spelling is almost the same, but the meaning is not, and the model only looks at meaning.


### 1.3 The neighbours of a word

`neighbours("clinic")` lists the words closest to `clinic`. Run it, then try
**three words of your own** (lowercase, and a common word, otherwise it may
not be in the list).

In [ ]:
print(neighbours("clinic"))

In [ ]:
print(neighbours("football"))
print(neighbours("pizza"))
print(neighbours("amsterdam"))


Your answer:

The neighbours make sense. `clinic` gives hospital, medical, nursing. `football` gives soccer, basketball, rugby (other sports). `pizza` gives sandwich, fries, burgers (other fast food). `amsterdam` gives rotterdam, paris, vienna (other cities). So words that are used in the same way end up close together.


### 1.4 Play Contexto (10 minutes, in the browser)

Open <https://contexto.me/en/> and, together with your neighbour, guess the
word of the day. After every guess the game tells you how *close* your word
is to the secret word. That is exactly the distance you computed above.

Then answer: how did you decide on your next guess?

Your answer:

We started with a broad word. When a word scored close, we guessed words that are used in the same situations (synonyms or things from the same topic). When a word scored far away, we switched to a completely different topic. So we moved step by step towards the secret word, just like the neighbours above.


### 1.5 A map of words

We cannot draw 50 numbers, so the cell below squashes them into 2 and draws a
map. Run it. Which words sit together? Which word is the odd one out?

In [ ]:
plot_words(["dog", "cat", "horse", "cow", "car", "bus", "train", "bike",
            "doctor", "nurse", "hospital", "clinic", "school", "teacher", "student",
            "king", "queen", "prince", "bread", "rice", "apple", "banana"])

Your answer:

The words form groups: animals (dog, cat, horse, cow), transport (car, bus, train, bike), health (doctor, nurse, hospital, clinic), school (school, teacher, student), royals (king, queen, prince) and food (bread, rice, apple, banana). A word that sits a bit apart from its group is the odd one out, for example `bike` or `cow`. The map is only a rough picture, because 50 numbers were squashed into 2.


### 1.6 Arithmetic with meaning

Because words are numbers, you can calculate with them.
`analogy("man", "king", "woman")` answers: *man is to king as woman is to ...?*

Run the first two. Then **predict** what `analogy("man", "doctor", "woman")`
will answer, and run it.

In [ ]:
print(analogy("man", "king", "woman"))
print(analogy("paris", "france", "berlin"))

Your prediction:

I think it gives `nurse`, because the internet texts often link doctor to men and nurse to women.


In [ ]:
print(analogy("man", "doctor", "woman"))
print(analogy("man", "manager", "woman"))

Your answer:

`man : king = woman : queen` and `paris : france = berlin : germany` work well. But `man : doctor = woman : ?` gives `nurse`, and `man : manager = woman : ?` gives `assistant`. This is a stereotype. The model learned it from texts written by people, so it copies the bias in those texts. If such a model is used to sort CVs, women could be treated unfairly.


### 1.7 One word, one list of numbers

Run the cell below. `bank` has **one** list of numbers, so it has one set of
neighbours.

In [ ]:
print(neighbours("bank"))

Your answer:

All neighbours of `bank` are about money (banks, banking, investment, credit). The river meaning is missing. The problem: an embedding gives a word only one list of numbers, so a word with two meanings gets the meaning it is used for most often. The context of the sentence is ignored.


---

**LEARNING WAVE 2 · SLIDES 9-14**

## 2. Order and memory: RNN and LSTM

Sections 2.1 to 2.7. Word counts throw away the order of the words. A recurrent network reads the words one by one and carries a memory along. How long can it keep that memory?

### 2.1 Order matters

Bag of words and plain embeddings treat a text as a pile of words. Run the
cell below: it compares the word counts of two sentences with opposite
meanings.

In [ ]:
from collections import Counter

good = "the food was good not bad at all"
bad = "the food was bad not good at all"

print(Counter(good.split()))
print(Counter(bad.split()))
print("Same counts?", Counter(good.split()) == Counter(bad.split()))

Your answer:

The counts are exactly the same (`Same counts? True`), but the sentences mean the opposite. So counting words throws away the order, and the order is what carries the meaning here (`good not bad` vs `bad not good`).


### 2.2 Telephone game (in class, 6 minutes, no laptop)

Your teacher will explain the rules. You play two rounds:

1. **The plain chain (the RNN).** A message is whispered down the line. Everyone only knows what the person before them whispered.
2. **The note-taking chain (the LSTM).** Everyone gets a small notebook and may write down one important thing from the message, keep it, and pass it on next to the whisper.

Write down what you notice: what got lost in round 1, and what did the
notebook change in round 2?

Your answer:

Round 1 (RNN): the message changed more and more along the line. The start of the message and details got lost, because everyone only heard the person before them.
Round 2 (LSTM): with the notebook the most important thing was kept until the end of the line. The whisper still changed, but the key fact survived. That is what the memory of an LSTM does.


### 2.3 The memory test

A tiny game to measure memory. Every **sequence** starts with a secret
colour word (red, green, blue or yellow), followed by a number of filler
words. The model reads the whole sequence, word by word, and then has to say
**which colour the sequence started with**.

The only way to win is to remember the first word until the end.

The cell below prepares two tiny models: an **RNN** and an **LSTM**. It also
contains the function `memory_test(kind, length)`, which trains a model for
a few seconds and returns its score on new sequences (1.0 = always right,
0.25 = just guessing). Run it, then look at two examples.

In [ ]:
# Run, do not read: two tiny models and a game to test their memory
import torch
import torch.nn as nn

colours = ["red", "green", "blue", "yellow"]
fillers = ["the", "cat", "sat", "on", "a", "mat", "and", "looked", "at", "dog"]

def make_data(n, length, generator):
    first = torch.randint(0, 4, (n,), generator=generator)
    filler = torch.randint(4, 14, (n, length), generator=generator)
    return torch.cat([first.unsqueeze(1), filler], dim=1), first

class TinyReader(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.embedding = nn.Embedding(14, 16)
        if kind == "rnn":
            self.reader = nn.RNN(16, 32, batch_first=True)
        else:
            self.reader = nn.LSTM(16, 32, batch_first=True)
            for name, p in self.reader.named_parameters():
                if "bias" in name:
                    p.data[32:64] = 1.0     # start with 'keep what you remember'
        self.answer = nn.Linear(32, 4)

    def forward(self, x):
        out, _ = self.reader(self.embedding(x))
        return self.answer(out[:, -1, :])    # the answer comes after the LAST word

def memory_test(kind, length, seed=1):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    x_train, y_train = make_data(2000, length, g)
    x_test, y_test = make_data(1000, length, g)
    net = TinyReader(kind)
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    loss_function = nn.CrossEntropyLoss()
    for epoch in range(30):
        order = torch.randperm(2000)
        for i in range(0, 2000, 100):
            batch = order[i:i + 100]
            optimizer.zero_grad()
            loss_function(net(x_train[batch]), y_train[batch]).backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
    with torch.no_grad():
        return round((net(x_test).argmax(1) == y_test).float().mean().item(), 2)

def show_example(length):
    g = torch.Generator().manual_seed(length)
    x, y = make_data(1, length, g)
    tokens = x[0].tolist()
    print(colours[tokens[0]], " ".join(fillers[t - 4] for t in tokens[1:]))
    print("-> the model must answer:", colours[int(y[0])])
    print()

In [ ]:
show_example(5)
show_example(15)

**Predict first.** We test sequences with 5, 10, 20 and 30 filler words
after the colour. For each length, will the model still remember the
colour? Write your guess for the RNN and for the LSTM.

Your prediction:

RNN: good at 5 words, worse at 10, around guessing (0.25) at 20 and 30.
LSTM: good at all lengths, maybe a bit worse at 30.


### 2.4 Test the RNN

This trains a fresh RNN for each length (about 5 seconds each) and prints
the score.

In [ ]:
lengths = [5, 10, 20, 30]
rnn_scores = []
for length in lengths:
    score = memory_test("rnn", length)
    rnn_scores.append(score)
    print("RNN, after", length, "filler words ->", score)

### 2.5 Test the LSTM

Exactly the same game, with the LSTM instead.

In [ ]:
lstm_scores = []
for length in lengths:
    score = memory_test("lstm", length)
    lstm_scores.append(score)
    print("LSTM, after", length, "filler words ->", score)

### 2.6 Compare

Run the cell below to draw both results in one picture, then answer.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(lengths, rnn_scores, marker="o", label="RNN")
plt.plot(lengths, lstm_scores, marker="o", label="LSTM")
plt.axhline(0.25, color="grey", linestyle="--", label="just guessing")
plt.xlabel("number of words between the secret word and the answer")
plt.ylabel("score (1.0 = always right)")
plt.legend()
plt.show()

Your answer:

The RNN is perfect at 5 and 10 words (1.0), but drops to 0.49 at 20 words and 0.25 at 30 words, which is just guessing. It forgets the first word. The LSTM scores 1.0 at every length, because its memory lane keeps the colour until the end. So an LSTM remembers much longer than a plain RNN.


### 2.7 Your turn: push the LSTM

Test the LSTM on a much longer sequence, for example 50 filler words. Use
`memory_test("lstm", 50)` and print the result. Do it a second time with
`seed=2` as an extra argument (`memory_test("lstm", 50, seed=2)`).

In [ ]:
print("LSTM, 50 filler words, seed 1 ->", memory_test("lstm", 50))
print("LSTM, 50 filler words, seed 2 ->", memory_test("lstm", 50, seed=2))


Your answer:

With 50 filler words the LSTM drops to 0.49 (seed 1) and 0.58 (seed 2). That is still better than guessing (0.25), but far from perfect, and the score changes with a different random start. So the LSTM remembers longer than an RNN, but very long texts are still hard. This is one reason transformers were invented.


---

**LEARNING WAVE 3 · SLIDES 15-20**

## 3. Context and attention: transformers

Sections 3.1 to 3.7. A transformer does not read one word at a time: every word looks at every other word at once. We use two small pre-trained transformers.

### 3.0 Load the transformers

Run this cell once. It downloads two small pre-trained models:
**DistilBERT**, which has been taught to read English (and, in a second
version, to judge whether a text is positive or negative), and **DistilGPT-2**, which has been taught to predict the next
word. This takes up to a minute. Ignore warnings about tokens.

In [ ]:
# Run, do not read: loads two small pre-trained transformers
from transformers import pipeline, AutoModel, AutoModelForCausalLM, AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment = pipeline("sentiment-analysis", model=checkpoint)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = AutoModel.from_pretrained("distilbert-base-uncased").eval()

lm_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
lm = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()

def show(sentences):
    for result, sentence in zip(sentiment(sentences), sentences):
        print(result["label"], round(result["score"], 2), "|", sentence)

def word_vector(sentence, word):
    # the list of numbers for ONE word, as this model sees it IN THIS sentence
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = encoder(**enc).last_hidden_state[0]
    position = enc["input_ids"][0].tolist().index(tokenizer.convert_tokens_to_ids(word))
    v = hidden[position]
    return v / v.norm()

def context_similarity(sentence_1, sentence_2, word="bank"):
    return round(float(word_vector(sentence_1, word) @ word_vector(sentence_2, word)), 2)

def next_words(prompt, n=6):
    ids = lm_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = lm(**ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, n)
    for p, i in zip(top.values, top.indices):
        print(round(float(p), 3), "|", lm_tokenizer.decode(int(i)))

def write(prompt, temperature=1.0, length=25, seed=1):
    torch.manual_seed(seed)
    ids = lm_tokenizer(prompt, return_tensors="pt")
    out = lm.generate(**ids, max_new_tokens=length, do_sample=True,
                      temperature=temperature, pad_token_id=lm_tokenizer.eos_token_id)
    print(lm_tokenizer.decode(out[0]).replace("\n", " "))

def pronoun_odds(prompt):
    ids = lm_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(lm(**ids).logits[0, -1], dim=-1)
    she = float(probs[lm_tokenizer.encode(" she")[0]])
    he = float(probs[lm_tokenizer.encode(" he")[0]])
    print(prompt, "| she:", round(she, 3), "| he:", round(he, 3))

### 3.1 One word, many meanings

In 1.7 `bank` had one list of numbers, always. A transformer gives a word a
**different list of numbers in every sentence**, because it looks at the
other words first.

`context_similarity(sentence_1, sentence_2)` compares the lists of numbers
of the word `bank` in two sentences. **Predict first:** which comparison
will be the *highest* and which the *lowest*?

- A: `she sat on the bank of the river`
- B: `he deposited money at the bank`
- C: `the bank approved the loan`
- D: `we had a picnic on the river bank`

Your prediction:

Highest: A and D (both about a river) or B and C (both about money). Lowest: a river sentence with a money sentence (A-B or D-C).


In [ ]:
A = "she sat on the bank of the river"
B = "he deposited money at the bank"
C = "the bank approved the loan"
D = "we had a picnic on the river bank"

print("A and D (both river):", context_similarity(A, D))
print("B and C (both money):", context_similarity(B, C))
print("A and B (river / money):", context_similarity(A, B))
print("D and C (river / money):", context_similarity(D, C))

Your answer:

My prediction was right. The two river sentences (A-D: 0.78) and the two money sentences (B-C: 0.78) score high. The mixed pairs score lower (A-B: 0.56, D-C: 0.54). So the transformer gives `bank` a different list of numbers depending on the other words in the sentence. This fixes the problem from 1.7, where `bank` only had one meaning.


### 3.2 Be attention (in class, 6 minutes)

Your teacher shows a sentence on screen: *The animal didn't cross the street
because it was too tired.* Every student is one word. The student who is `it`
points at the words they need to look at to know what `it` means, and the
rest of the class checks.

Then the teacher changes the end of the sentence to *... because it was too
wide.* Who does `it` look at now?

Write down: which words did `it` look at, and what changed?

Your answer:

With "because it was too **tired**", `it` looks at `animal`, because an animal can be tired and a street cannot. With "because it was too **wide**", `it` looks at `street`, because a street can be wide. Only one word changed, but `it` now points at another word. This is attention: every word looks at the other words to decide what it means.


### 3.3 Sentiment: where does it fail?

`show([...])` gives you the model's verdict (POSITIVE or NEGATIVE) and how
sure it is (0 to 1) for a list of sentences. Look at the sentences below and
**predict** which ones the model will get wrong.

Your prediction:

I think the model gets these wrong: the double negative ("wouldn't say ... unfriendly"), the sarcastic one ("Great, another two-hour wait"), the typo sentence ("the cluic wuz rly good") and the Dutch sentence. The mixed one (kind doctor, awful waiting room) is also hard because it is both.


In [ ]:
show(["The clinic helped me a lot.",
      "The clinic did not help me at all.",
      "I wouldn't say the staff were unfriendly.",
      "Great, another two-hour wait. Just what I needed.",
      "The doctor was kind, but the waiting room was awful.",
      "the cluic wuz rly good",
      "Het was een heel goede ervaring."])

Your answer:

The model got 3 sentences wrong, and was very sure each time:
- the double negative "I wouldn't say the staff were unfriendly" -> NEGATIVE (0.97), but it is positive;
- the sarcastic "Great, another two-hour wait" -> POSITIVE (0.99), but it is negative;
- the Dutch "Het was een heel goede ervaring" -> NEGATIVE (0.97), but it is positive. The model only knows English.

Surprise: the typo sentence "the cluic wuz rly good" was correct (POSITIVE, 1.0), so I was wrong there. The mixed sentence got NEGATIVE, which is only half the truth. The big problem: the model is almost 100% sure, even when it is wrong.


Now try **three sentences of your own** that you think will fool the
model. Use `show(["sentence one", "sentence two", "sentence three"])`.

In [ ]:
show(["Oh wonderful, my appointment got cancelled again.",
      "The nurse was not unhelpful.",
      "Die dokter was echt top, ik voel me veel beter."])


Your answer:

1. Sarcasm ("Oh wonderful, my appointment got cancelled again") -> NEGATIVE (1.0). Correct! The model was not fooled, probably because "cancelled again" is clearly negative.
2. Double negative ("The nurse was not unhelpful") -> NEGATIVE (0.53). Wrong: it means mildly positive. The model is unsure here (0.53 is almost a coin flip).
3. Dutch ("Die dokter was echt top, ik voel me veel beter") -> NEGATIVE (0.98). Wrong, and very sure. The model does not understand Dutch.

So 2 of my 3 sentences fooled the model. Dutch fails every time.


### 3.4 What an LLM does: guess the next word

ChatGPT, Claude and Gemini are transformers trained to do one thing:
**predict the next word**. `next_words(prompt)` shows the six most likely
next words (with their probability) according to DistilGPT-2, a small
relative. Run it, then try your own prompt.

In [ ]:
next_words("The doctor said that")

In [ ]:
next_words("The patient felt")


Your answer:

For "The doctor said that" the top next words are "he" (0.174) and "the" (0.159). "she" is much lower (0.044). For "The patient felt" the top words are "the", "a", "that", "like", "he" and "pain", all with low chances. The model does not know what is true. It only picks words that often came next in its training texts. That is all an LLM does: guess the next word, again and again.


### 3.5 Temperature

`write(prompt, temperature=...)` lets the model keep adding words. The
**temperature** sets how adventurous it is: low = always take the most likely
next word, high = sometimes pick an unlikely one. Run both and compare.

In [ ]:
write("The doctor said that", temperature=0.3)
print()
write("The doctor said that", temperature=1.5)

Your answer:

With temperature 0.3 the text is normal and makes sense ("the patient was not in the hospital for the first time ..."). With temperature 1.5 it starts fine but becomes strange ("I was about to fall behind the rate in my primary care system"), because the model picks more unlikely words. Low temperature = safe and predictable, high temperature = creative but more nonsense.


### 3.6 What did it learn from the internet?

A language model learned from texts written by people. The cell below asks
the model how likely the next word is **she** or **he** after four different
jobs.

In [ ]:
pronoun_odds("The nurse said that")
pronoun_odds("The engineer said that")
pronoun_odds("The doctor said that")
pronoun_odds("The teacher said that")

Your answer:

- nurse: she 0.224, he 0.032 -> "she" is 7 times more likely.
- engineer: she 0.011, he 0.109 -> "he" is 10 times more likely.
- doctor: she 0.044, he 0.174 -> "he" is 4 times more likely.
- teacher: she 0.162, he 0.147 -> almost equal.

The model learned this from internet texts, where these jobs are often written about this way. So it copies the stereotype, just like the embeddings in 1.6.


### 3.7 Who is misread?

You saw three kinds of failure today: a stereotype in the embedding (1.6),
a sarcastic sentence (3.3) and a Dutch sentence (3.3). Imagine a clinic's
helpline that uses a language model to sort incoming messages into *urgent*
and *not urgent*.

Which people would be misread most often, and what would a responsible
clinic do about it?

Your answer:

People who would be misread most:
- people who write in another language (for example Dutch, Arabic or Turkish), or mix languages;
- people who make many typos or use slang, for example people with dyslexia or low literacy;
- people who use sarcasm or understate their problem ("I'm fine, just can't breathe very well").

So the people who already have less access to care would be helped last. A responsible clinic would: keep a human who checks the messages (at least the "not urgent" ones), test the model on these groups first, use a multilingual model, and give patients a phone number or button for real emergencies so they never depend only on the model.


---

## 4. Big individual assignment: stress-test a language model

**You are a tester, not a builder.** An organisation wants to use the
sentiment model from section 3 to read feedback from its users. Before it
does, it asks you: **where does this model fail, and who would that hurt?**

Pick a context you care about (a clinic, a food bank, a housing association,
a school, a municipality ...). You work with the pre-trained model `sentiment`
and the `show` function from section 3.

**Required skills and outputs**

1. **A test set of at least 12 sentences** about your context, covering at
   least four of these kinds: negation, mixed feelings, sarcasm or irony,
   a long sentence where the key word is far from the end, typos or slang,
   a sentence in another language (for example Dutch).
2. **Your own label first.** Before running anything, write down for each
   sentence what a person would say (positive or negative).
3. **Run the model** and collect the results in a table (a pandas
   DataFrame): sentence, your label, the model's label, its confidence, and
   whether they agree.
4. **Error analysis.** Which kinds of sentences fail? For three failures, say
   what a reader would have needed to get it right (the order of the words,
   a memory of an earlier word, the context around a word) and which step of
   today's ladder (embedding, RNN, LSTM, transformer) would or would not
   have handled it, and why.
5. **A bias probe.** Use `next_words` or `pronoun_odds` with at least two
   prompts that differ in one word (for example two jobs, two names or two
   nationalities) and describe what changes.
6. **A written judgement** (about 150 words): would you let this model read
   the users' feedback? Under which conditions? Who is misread, and what
   does the organisation do about them?

**Bonus options**

- Run your test set through a second model from <https://huggingface.co/models>
  (search for a multilingual sentiment model) by creating a second
  `pipeline("sentiment-analysis", model="...")`, and compare the table.
- Try three different temperatures with `write` for one story prompt and
  describe how the text changes.
- Find a biased analogy in the embeddings of section 1 with `analogy`.

You are graded on the honesty of your error analysis and on your reasoning
about harm, not on how many sentences the model gets right.

In [ ]:
# 1. Context I chose:
# A local food bank (voedselbank) that wants to read feedback from its clients automatically.

import pandas as pd

# 2. My test sentences and my own labels (written BEFORE running the model)
tests = [
    # (sentence, my label, kind)
    ("The volunteers were friendly and helped me quickly.", "POSITIVE", "simple"),
    ("The food was not fresh at all.", "NEGATIVE", "negation"),
    ("I can't say the staff were rude.", "POSITIVE", "negation"),
    ("The volunteers were kind, but the queue outside in the rain was terrible.", "NEGATIVE", "mixed"),
    ("The package was small, but it really helped my family this week.", "POSITIVE", "mixed"),
    ("Oh great, they ran out of bread again. Lovely.", "NEGATIVE", "sarcasm"),
    ("Wow, only three hours of waiting, what a treat.", "NEGATIVE", "sarcasm"),
    ("When I first came here I was nervous and ashamed, and I had heard many bad stories from neighbours, but in the end it was okay.", "POSITIVE", "long"),
    ("Everything about the building, the volunteers and the opening times was great, except that the food made my kids sick.", "NEGATIVE", "long"),
    ("da food was rly gud tbh", "POSITIVE", "typos/slang"),
    ("the vollunteers wer so rood to me", "NEGATIVE", "typos/slang"),
    ("De vrijwilligers waren heel aardig.", "POSITIVE", "Dutch"),
    ("Het eten was bedorven en ik voelde me niet welkom.", "NEGATIVE", "Dutch"),
]

# 3. Run the model, build the table
sentences = [t[0] for t in tests]
results = sentiment(sentences)
table = pd.DataFrame({
    "sentence": sentences,
    "kind": [t[2] for t in tests],
    "my label": [t[1] for t in tests],
    "model label": [r["label"] for r in results],
    "confidence": [round(r["score"], 2) for r in results],
})
table["agree"] = table["my label"] == table["model label"]
pd.set_option("display.max_colwidth", 80)
display(table)
print("Correct:", table["agree"].sum(), "of", len(table))
print()
print(table.groupby("kind")["agree"].mean())

# 4. Which sentences failed? (see the markdown cell below for the explanation)
print()
print("Failures:")
display(table[~table["agree"]])

# 5. Bias probe (prompts that differ in one word)
print()
pronoun_odds("The volunteer said that")
pronoun_odds("The manager said that")
print()
next_words("The Dutch man at the food bank was")
print()
next_words("The Moroccan man at the food bank was")

# 6. Written judgement: see the markdown cell below


Your answer:

**4. Error analysis**

- **Sarcasm** ("Oh great, they ran out of bread again. Lovely."): the model sees "great" and "lovely" and says POSITIVE. A reader needs the **context around the words**: running out of bread is bad, so "great" is meant the other way. Embeddings and an RNN/LSTM cannot do this, because they read the word "great" as positive. A transformer looks at all words at once, so in theory it can, but this small model has seen little sarcasm, so it still fails.
- **Dutch** ("Het eten was bedorven ..."): the model was only trained on English, so it does not know these words at all. No step of the ladder helps here if the training data has no Dutch. You need a model trained on Dutch (a multilingual model).
- **Long sentence with the key word far away** ("Everything ... was great, except that the food made my kids sick"): many positive words, and the important part comes at the end. A reader needs the **order** and a **memory** of the whole sentence. Embeddings (no order) fail. An RNN forgets the start; an LSTM does better. A transformer can look at "except" and "sick" together, so it has the best chance.
- **Double negative** ("I can't say the staff were rude"): the model sees "can't" and "rude" and says NEGATIVE. You need the **order** of the words. Bag of words and embeddings cannot see this; an LSTM or transformer can, if it saw enough examples.

**5. Bias probe**

"The volunteer said that" vs "The manager said that": only the job changes, but the odds of "she" vs "he" change, with "he" more likely for manager. With "Dutch man" vs "Moroccan man", the predicted next words can be different, and sometimes more negative for one group. The model copies the stereotypes from its internet training texts.

**6. Written judgement**

I would not let this model read the food bank's feedback on its own. It works on simple English sentences, but it fails on exactly the way many food bank clients write: in Dutch or another language, with typos, with sarcasm when they are frustrated, or with mixed feelings. And it is very sure of itself even when it is wrong. That means the people who struggle the most, like non-native speakers and people with low literacy, are misread the most, and their complaints could be missed. The organisation could use it only under conditions: a human reads all feedback that is negative, unclear or not in English; they test it on real client messages first; they use a multilingual model; and clients can always talk to a person. The model should help sort, not decide.


---

## 5. Wrap-up

### 5.1 The ladder

Each step fixed a weakness of the one before:

| Model | What it does well | What it still cannot do |
|-------|-------------------|-------------------------|
| Bag of words | counts which words occur, simple and fast | no order, no meaning |
| **Word embeddings** | words that are used alike get similar numbers | one fixed meaning per word, no order |
| **RNN** | reads in order, any length | forgets the beginning of long texts |
| **LSTM** | keeps what matters in a memory lane | reads one word at a time, slow, long texts still hard |
| **Transformer** | every word looks at every other word at once, meaning depends on the context | needs huge data and computers; copies the bias of its data |

### 5.2 Five things to remember

1. A computer does not read. It turns text into numbers and calculates.
2. Embeddings give every word a place on a map of meaning, so similar words end up close.
3. Order matters. An RNN reads in order but forgets; an LSTM remembers longer.
4. A transformer lets every word look at every other word. That is why context finally works, and it is the foundation of ChatGPT, Claude and Gemini.
5. All of these models copy the data they learned from. They sound sure even when they are wrong, so keep a human in the loop and ask who is misread.

### 5.3 Homework (while you work on the hackathon)

Complete these DataCamp chapters:

1. *Recurrent Neural Networks (RNN) for Language Modeling with Keras*: **Chapter 1** and **Chapter 4**
   <https://app.datacamp.com/learn/courses/recurrent-neural-networks-rnn-for-language-modeling-with-keras>
2. *Introduction to LLMs in Python*: the whole chapter **Getting started with Large Language Models (LLMs)**
   <https://campus.datacamp.com/courses/introduction-to-llms-in-python/getting-started-with-large-language-models-llms>

Also finish section 4 and upload the notebook to `Term 1/Week 6/homework/` in your portfolio.

**Next week:** Computer Vision
**Friday:** the hackathon. You find out on the day which tool you use and which problem you solve.